# 07 - Liwa Oasis Spatial-Temporal Change & Anomaly Detection

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook performs spatial-temporal differencing and anomaly detection over Sentinel-2 scenes in Liwa Oasis to map vegetation stress degradation and soil salinity expansion.

The mechanistic justification for this step is: Statistical differencing of multi-temporal index maps ($\Delta \text{NDVI}$, $\Delta \text{SI}$) isolates degraded date palm sectors for ground-probe deployment.

In [ ]:
# Step 1: Environment Setup & Multi-Temporal Differencing
# The mechanistic justification for this step is: Differencing early vs late season Sentinel-2 scenes computes spatial delta maps.
import os
import glob
import re
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import rasterio

def find_files(pattern):
    return sorted(glob.glob(pattern, recursive=True))

s2_b04_files = find_files("/kaggle/input/**/B04.tif") or find_files("../data/**/B04.tif")
s2_b08_files = find_files("/kaggle/input/**/B08.tif") or find_files("../data/**/B08.tif")

if len(s2_b04_files) >= 2 and len(s2_b08_files) >= 2:
    # Group B04 and B08 files by UTM tile identifier (e.g., 39QYF from parent directory name)
    b08_by_dir = {os.path.dirname(f): f for f in s2_b08_files}
    tile_groups = defaultdict(list)
    for idx, b04_path in enumerate(s2_b04_files):
        parent_dir = os.path.dirname(b04_path)
        parent_name = os.path.basename(parent_dir)
        b08_path = b08_by_dir.get(parent_dir, s2_b08_files[idx] if idx < len(s2_b08_files) else None)
        if not b08_path:
            continue
        tile_match = re.search(r'(\d{2}[A-Z]{3})', parent_name)
        tile_id = tile_match.group(1) if tile_match else "UNKNOWN"
        date_match = re.search(r'(\d{8})', parent_name)
        date_str = date_match.group(1) if date_match else parent_name
        tile_groups[tile_id].append({
            'b04': b04_path,
            'b08': b08_path,
            'scene': parent_name,
            'date': date_str
        })

    # Find a UTM tile that has at least 2 scenes at different dates
    selected_tile = None
    target_scenes = None
    for tile_id in sorted(tile_groups.keys()):
        scenes = sorted(tile_groups[tile_id], key=lambda s: (s['date'], s['scene']))
        unique_dates = {s['date'] for s in scenes}
        if len(unique_dates) >= 2:
            selected_tile = tile_id
            target_scenes = (scenes[0], scenes[-1])
            break

    if target_scenes is not None:
        scene_t1, scene_t2 = target_scenes
        b04_t1, nir_t1 = scene_t1['b04'], scene_t1['b08']
        b04_t2, nir_t2 = scene_t2['b04'], scene_t2['b08']
        print(f"Selected UTM tile '{selected_tile}' with multi-temporal coverage:")
        print(f"  t1 (earliest): {scene_t1['scene']} (Date: {scene_t1['date']})")
        print(f"  t2 (latest):   {scene_t2['scene']} (Date: {scene_t2['date']})")
    else:
        # Limitation: No single UTM tile has multi-temporal coverage across different dates.
        # Differencing scenes across different UTM tiles or identical dates has geographic/temporal limitations,
        # but using the two closest scenes as fallback for pipeline continuity.
        print("Diagnostic: No UTM tile with multiple acquisition dates found. Falling back to two closest scenes (s2_b04_files[0], s2_b04_files[1]).")
        b04_t1, nir_t1 = s2_b04_files[0], s2_b08_files[0]
        b04_t2, nir_t2 = s2_b04_files[1], s2_b08_files[1]

    with rasterio.open(b04_t1) as r1, rasterio.open(nir_t1) as nir1, \
         rasterio.open(b04_t2) as r2, rasterio.open(nir_t2) as nir2:
        
        red1, nir1 = r1.read(1).astype(np.float32) / 10000.0, nir1.read(1).astype(np.float32) / 10000.0
        red2, nir2 = r2.read(1).astype(np.float32) / 10000.0, nir2.read(1).astype(np.float32) / 10000.0
        
        min_h = min(red1.shape[0], red2.shape[0])
        min_w = min(red1.shape[1], red2.shape[1])
        
        ndvi1 = (nir1[:min_h, :min_w] - red1[:min_h, :min_w]) / (nir1[:min_h, :min_w] + red1[:min_h, :min_w] + 1e-8)
        ndvi2 = (nir2[:min_h, :min_w] - red2[:min_h, :min_w]) / (nir2[:min_h, :min_w] + red2[:min_h, :min_w] + 1e-8)
        
        delta_ndvi = ndvi2 - ndvi1
        
        print(f"Delta NDVI Range: Min={np.nanmin(delta_ndvi):.4f}, Max={np.nanmax(delta_ndvi):.4f}, Mean={np.nanmean(delta_ndvi):.4f}")
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        im0 = axes[0].imshow(delta_ndvi, cmap='RdYlGn', vmin=-0.3, vmax=0.3)
        axes[0].set_title("Sentinel-2 Delta NDVI (t2 - t1)")
        plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
        
        # Anomaly Mask (Degradation > 2 std dev)
        threshold = np.nanmean(delta_ndvi) - 2 * np.nanstd(delta_ndvi)
        anomaly_mask = delta_ndvi < threshold
        
        im1 = axes[1].imshow(anomaly_mask, cmap='Reds')
        axes[1].set_title(f"Vegetation Stress Anomaly Mask (Delta < {threshold:.3f})")
        plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
        
        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print("Multiple Sentinel-2 scenes not mounted. Skipping change detection...")
